# Used Car Price Prediction

The goal of this project is to develop a machine learning model that can predict the price of a used car, and examine the characteristics most strongly associated with
pricing.

Four regression models, Linear Regression, Ridge Regression, Decision Tree, and Random Forest, are developed and
compared using cross-validation. MAE will be the primary
metric for model selection. R^2 will also be evaluated on the final model to determine how much the model accounts for variation in price.

The champion model is evaluated on a held-out test set, followed by permutation importance to determine what features the model relies on most heavily to predict price.

## Imports and Data Loading

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.inspection import permutation_importance

In [ ]:
df = pd.read_csv("used_cars.csv")

In [3]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


## Initial Data Inspection

In [4]:
#Examining data types and checking for null values.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 376.0 KB


In [5]:
#Checking for duplicate rows.
df.duplicated().sum()

np.int64(0)

There are no duplicated rows. Fuel type, accident, and clean title columns all contain null values. Price and milage are stored as strings and need to be converted to numeric values.

In [6]:
df['price'] = pd.to_numeric(
    df['price']
        .str.replace('$',"", regex=False)
        .str.replace(',',"", regex=False),
    errors='coerce'
)

df['milage'] = pd.to_numeric(
    df['milage']
        .str.replace(' mi.',"",regex=False)
        .str.replace(',',"",regex=False)
    )

Now that all columns contain the correct data types, I am going to examine the categorical columns.

In [7]:
#Examining number of unique values for each variable.
df.nunique()

brand             57
model           1898
model_year        34
milage          2818
fuel_type          7
engine          1146
transmission      62
ext_col          319
int_col          156
accident           2
clean_title        1
price           1569
dtype: int64

In [8]:
#Separating categorical columns to inspect frequently occuring values.
cat_cols = df.select_dtypes(include='str').columns.to_list()

for col in cat_cols:
    counts = df[col].value_counts(ascending=False).head(8)
    print(counts)

brand
Ford             386
BMW              375
Mercedes-Benz    315
Chevrolet        292
Porsche          201
Audi             200
Toyota           199
Lexus            163
Name: count, dtype: int64
model
M3 Base               30
F-150 XLT             24
Corvette Base         22
1500 Laramie          18
Model Y Long Range    17
Camaro 2SS            17
Wrangler Sport        17
Mustang GT Premium    16
Name: count, dtype: int64
fuel_type
Gasoline          3309
Hybrid             194
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64
engine
2.0L I4 16V GDI DOHC Turbo                               52
355.0HP 5.3L 8 Cylinder Engine Gasoline Fuel             48
420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel             47
–                                                        45
300.0HP 3.0L Straight 6 Cylinder Engine Gasoline Fuel    44
240.0HP 2.0L 4 Cylinder Engine Gasoline Fuel             42
285.

## Data Cleaning

In [9]:
df['fuel_type'].value_counts()

fuel_type
Gasoline          3309
Hybrid             194
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

In [10]:
df[df['fuel_type']=='–']

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
130,Chrysler,Pacifica Touring,2017,87305,–,–,9-Speed A/T,Silver,Black,None reported,Yes,9000
257,Toyota,Land Cruiser Base,1993,231500,–,–,A/T,White,Gray,None reported,Yes,29500
338,Mazda,Mazda3 s Grand Touring,2016,85000,–,–,A/T,Black,Black,At least 1 accident or damage reported,Yes,18500
491,Chrysler,Pacifica Touring,2017,63000,–,–,9-Speed A/T,White,Beige,None reported,Yes,20000
692,Honda,Civic EX,1993,58981,–,–,A/T,White,Beige,None reported,Yes,9599
855,Ford,Bronco,1974,6217,–,–,–,Dark Gray Metallic,–,None reported,Yes,115000
923,Dodge,Challenger R/T,2010,100100,–,–,Transmission w/Dual Shift Mode,Purple,Black,None reported,Yes,17000
995,Dodge,Challenger R/T Scat Pack,2019,43000,–,–,6-Speed M/T,Green,Black,None reported,Yes,50000
1063,Mazda,Mazda6 i Grand Touring,2016,83100,–,–,A/T,White,White,None reported,Yes,17800
1083,Dodge,Challenger R/T,2010,73000,–,–,A/T,Black,Black,None reported,Yes,17500


In [11]:
df[df['engine']=='–'].shape

(45, 12)

In [12]:
df = df[df['fuel_type']!='–']

Confirmed that all rows with '-' fuel type also had '-' as the engine. Dropped rows with '-' listed as the engine and fuel type, since they only account for ~1.1% of the data set.

In [13]:
df[df['fuel_type']=='not supported']

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
2894,Toyota,Mirai Limited,2021,44000,not supported,182.0HP Electric Motor Hydrogen Fuel,A/T,Gray,Black,None reported,Yes,18500
3700,Toyota,Mirai Base,2016,40000,not supported,151.0HP Electric Motor Hydrogen Fuel,A/T,Silver,Black,None reported,Yes,9500


In [14]:
#Checking the null values in the fuel type column.
df[df['fuel_type'].isna()]

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
9,Tesla,Model X Long Range Plus,2020,34000,NaN,534.0HP Electric Motor Electric Fuel System,A/T,Black,Black,None reported,Yes,69950
44,Lucid,Air Grand Touring,2022,3552,NaN,536.0HP Electric Motor Electric Fuel System,1-Speed A/T,Red,Beige,None reported,Yes,119999
68,Lucid,Air Grand Touring,2022,4900,NaN,536.0HP Electric Motor Electric Fuel System,1-Speed A/T,Red,Black,None reported,Yes,99000
92,Rivian,R1S Adventure Package,2023,2800,NaN,835.0HP Electric Motor Electric Fuel System,1-Speed A/T,White,Green,None reported,Yes,92000
122,Rivian,R1S Adventure Package,2023,2500,NaN,835.0HP Electric Motor Electric Fuel System,A/T,Green,White,None reported,Yes,94000
...,...,...,...,...,...,...,...,...,...,...,...,...
3908,Tesla,Model Y Long Range,2021,49252,NaN,Electric,1-Speed Automatic,Deep Blue Metallic,Black,None reported,NaN,37999
3913,Tesla,Model S 100D,2018,65000,NaN,518.0HP Electric Motor Electric Fuel System,A/T,Blue,Black,None reported,Yes,41000
3932,Tesla,Model X Performance,2020,25535,NaN,778.0HP Electric Motor Electric Fuel System,A/T,White,Beige,None reported,Yes,68969
3961,Tesla,Model 3 Long Range,2021,23749,NaN,Electric,1-Speed Automatic,White,Black,None reported,NaN,35999


In [15]:
df[df['fuel_type'].isna()].value_counts('brand')

brand
Tesla            87
Rivian           17
Ford             12
Porsche           9
Audi              6
Chevrolet         5
Nissan            5
BMW               5
Kia               4
Lucid             3
Mercedes-Benz     3
Hyundai           3
Volkswagen        3
Karma             2
Polestar          1
FIAT              1
Volvo             1
Toyota            1
Cadillac          1
GMC               1
Name: count, dtype: int64

In [16]:
df[(df['fuel_type'].isna()) & (df['brand']!='Tesla')]

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
44,Lucid,Air Grand Touring,2022,3552,NaN,536.0HP Electric Motor Electric Fuel System,1-Speed A/T,Red,Beige,None reported,Yes,119999
68,Lucid,Air Grand Touring,2022,4900,NaN,536.0HP Electric Motor Electric Fuel System,1-Speed A/T,Red,Black,None reported,Yes,99000
92,Rivian,R1S Adventure Package,2023,2800,NaN,835.0HP Electric Motor Electric Fuel System,1-Speed A/T,White,Green,None reported,Yes,92000
122,Rivian,R1S Adventure Package,2023,2500,NaN,835.0HP Electric Motor Electric Fuel System,A/T,Green,White,None reported,Yes,94000
129,Lucid,Air Pure,2023,1300,NaN,620.0HP Electric Motor Electric Fuel System,A/T,Black,Gray,NaN,NaN,86900
...,...,...,...,...,...,...,...,...,...,...,...,...
3768,GMC,HUMMER EV Edition 1,2023,940,NaN,Electric Motor Electric Fuel System,A/T,White,Black,NaN,NaN,146000
3783,Porsche,Taycan 4S,2022,7900,NaN,482.0HP Electric Motor Electric Fuel System,2-Speed A/T,Red,–,None reported,Yes,108000
3813,Ford,Mustang Mach-E GT,2023,9095,NaN,480.0HP Electric Motor Electric Fuel System,A/T,Orange,Gray,None reported,Yes,59000
3834,Ford,F-150 Lightning LARIAT,2022,470,NaN,563.0HP Electric Motor Electric Fuel System,1-Speed A/T,White,Gray,None reported,Yes,71900


In [17]:
df['fuel_type'] = df['fuel_type'].fillna('Electric')

Confirmed all the vehicles with null values for fuel type were electric. Replaced null values for these vehicles with 'Electric'.

In [18]:
df['transmission'].value_counts().head(10)

transmission
A/T                               1015
8-Speed A/T                        405
Transmission w/Dual Shift Mode     396
6-Speed A/T                        362
6-Speed M/T                        243
Automatic                          237
7-Speed A/T                        209
8-Speed Automatic                  176
10-Speed A/T                       119
5-Speed A/T                         85
Name: count, dtype: int64

In [19]:
df['transmission'] = (
    df['transmission']
    .str.strip()
    .str.replace('Automatic','A/T', regex=False)
    .str.replace('Manual','M/T', regex=False)
)

In [20]:
df['transmission'].value_counts().head(10)

transmission
A/T                               1252
8-Speed A/T                        581
6-Speed A/T                        435
Transmission w/Dual Shift Mode     396
6-Speed M/T                        251
7-Speed A/T                        216
10-Speed A/T                       175
9-Speed A/T                        119
5-Speed A/T                         94
1-Speed A/T                         78
Name: count, dtype: int64

Noticed that many of the transmissions had two different ways of expressing manual or automatic. Standardized for all speeds.

In [21]:
df['accident'] = df['accident'].fillna('Unknown')
df['clean_title'] = df['clean_title'].fillna('Unknown')

In [22]:
df['int_col'] = df['int_col'].replace('–','Unknown')
df['ext_col'] = df['ext_col'].replace('–','Unknown')

Replaced null and placeholder values for the remaining categorical variables with unknown since there is no way to determine the actual values. Continuing to inspection and cleaning of numerical variables.

In [23]:
df.describe()

,model_year,milage,price
count,3964.000000,3964.000000,3.964000e+03
mean,2015.672301,64357.393037,4.465317e+04
std,5.837338,52127.804437,7.858455e+04
min,1996.000000,100.000000,2.000000e+03
25%,2012.000000,22688.750000,1.750000e+04
50%,2017.000000,52337.500000,3.145000e+04
75%,2020.000000,93500.000000,4.999800e+04
max,2024.000000,405000.000000,2.954083e+06


In [24]:
df.sort_values('milage',ascending=False).head(10)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
2764,Honda,Accord Sport,2013,405000,Gasoline,185.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,CVT Transmission,Black,Black,None reported,Yes,6000
3348,Mercedes-Benz,Sprinter 2500,2016,399000,Diesel,188.0HP 3.0L V6 Cylinder Engine Diesel Fuel,5-Speed A/T,Silver,Unknown,At least 1 accident or damage reported,Yes,24500
2175,Ford,Excursion Limited,2001,333361,Diesel,250.0HP 7.3L 8 Cylinder Engine Diesel Fuel,A/T,Black,Beige,At least 1 accident or damage reported,Yes,17000
3757,Ford,F-250 XL Crew Cab Super Duty,2004,315000,Diesel,325.0HP 6.0L 8 Cylinder Engine Diesel Fuel,A/T,White,Gray,None reported,Yes,9200
31,Dodge,Ram 1500 Laramie Mega Cab,2006,300183,Gasoline,345.0HP 5.7L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Gray,At least 1 accident or damage reported,Yes,10900
1145,GMC,Sierra 3500 SLE,2002,300000,Diesel,300.0HP 6.6L 8 Cylinder Engine Diesel Fuel,5-Speed A/T,Black,Gray,At least 1 accident or damage reported,Yes,5000
3118,Dodge,Dakota Sport,1997,285000,Gasoline,230.0HP 5.2L 8 Cylinder Engine Gasoline Fuel,4-Speed A/T,White,Gray,None reported,Yes,6399
3747,Ford,F-150 Lariat SuperCrew,2004,271000,Gasoline,300.0HP 5.4L 8 Cylinder Engine Gasoline Fuel,A/T,Red,Beige,At least 1 accident or damage reported,Yes,9500
2558,Toyota,Land Cruiser Base,2010,269885,Gasoline,381.0HP 5.7L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,Black,Beige,At least 1 accident or damage reported,Yes,22500
3158,Toyota,4Runner 4WD,1999,267000,Gasoline,150.0HP 2.7L 4 Cylinder Engine Gasoline Fuel,5-Speed M/T,Silver,Beige,At least 1 accident or damage reported,Yes,8000


In [25]:
df.sort_values('price',ascending=False).head(10)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
693,Maserati,Quattroporte Base,2005,32000,Gasoline,394.0HP 4.2L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Red,Beige,At least 1 accident or damage reported,Yes,2954083
229,Bugatti,Veyron 16.4 Grand Sport,2011,6330,Gasoline,8.0L W16 64V GDI DOHC Twin Turbo,7-Speed A/T with Auto-Shift,White,White,None reported,Yes,1950995
3046,Porsche,Carrera GT Base,2005,4400,Gasoline,605.0HP 5.7L 10 Cylinder Engine Gasoline Fuel,6-Speed M/T,Gray,Black,None reported,Yes,1599000
1356,Lamborghini,Aventador SVJ Base,2021,6987,Gasoline,6.5L V12 48V MPFI DOHC,7-Speed,Unknown,Nero Ade,None reported,Yes,749950
624,Rolls-Royce,Cullinan,2022,398,Gasoline,6.7L V12 48V GDI DOHC Twin Turbo,8-Speed A/T,Silver,Charles Blue,None reported,Yes,695000
979,Lamborghini,Aventador SVJ Base,2019,6929,Gasoline,759.0HP 6.5L 12 Cylinder Engine Gasoline Fuel,A/T,Blue,Black,None reported,Yes,649999
1508,Rolls-Royce,Phantom,2018,7585,Gasoline,6.8L V12 48V GDI DOHC Twin Turbo,8-Speed A/T,Unknown,Unknown,None reported,Yes,599000
3655,Lamborghini,Aventador S Base,2018,5858,Gasoline,729.0HP 6.5L 12 Cylinder Engine Gasoline Fuel,7-Speed A/T,White,Gray,None reported,Yes,491836
1061,Dodge,Viper GTC,2017,1389,Gasoline,645.0HP 8.4L 10 Cylinder Engine Gasoline Fuel,M/T,Black,Black,None reported,Yes,489995
1148,Porsche,911 R,2016,719,Gasoline,4.0L H6 24V GDI DOHC,6-Speed M/T,White,Black / Brown,None reported,Yes,489000


In [26]:
df = df.drop(index=df['price'].idxmax())

There were a few higher milage values, but none that were unrealistic. The highest price value was inspected and dropped as it is a suspected data error.

# Feature Engineering

In [27]:
#Using 2024 as the reference year since the dataset is from 2024.
df['vehicle_age'] = 2024 - df['model_year']

In [28]:
#Extracting hp and liters from the engine descriptions.
df['hp'] = pd.to_numeric(
    df['engine'].str.extract(r"(?i)(\d+(?:\.\d+)?)\s*HP")[0],
    errors='coerce'
)

df['liters'] = pd.to_numeric(
    df['engine'].str.extract(r"(?i)(\d+(?:\.\d+)?)\s*l(?:iter)?s?\b")[0],
    errors='coerce'
)

df[['engine','hp','liters']].head()

,engine,hp,liters
0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,300.0,3.7
1,3.8L V6 24V GDI DOHC,NaN,3.8
2,3.5 Liter DOHC,NaN,3.5
3,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,354.0,3.5
4,2.0L I4 16V GDI DOHC Turbo,NaN,2.0


In [29]:
#Filling the null hp columns with the mean of the vehicle brand, and the null liters will the mode of the brand.
class BrandImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.hp_by_brand_ = X.groupby('brand')['hp'].mean()

        self.liters_by_brand_ = X.groupby('brand')['liters'].agg(
            lambda values: (
                values.mode().iloc[0]
                if not values.mode().empty
                else np.nan
            )
        )

        return self

    def transform(self, X):
        X = X.copy()

        X['hp'] = X['hp'].fillna(
            X['brand'].map(self.hp_by_brand_)
        )

        X['liters'] = X['liters'].fillna(
            X['brand'].map(self.liters_by_brand_)
        )

        return X

## Train Test Split

I reserved 20% of the data as a hold-out set for final testing. Model year was dropped as vehicle age will be used in its place.

In [30]:
X = df.drop(columns=['price','model_year'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Pipelines

In [31]:
categorical_cols = X.select_dtypes(include='str').columns.to_list()
numerical_cols = X.select_dtypes(include='number').columns.to_list()

In [32]:
numerical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median', add_indicator=True)),
    ('scalar',StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

column_preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_cols),
    ('categorical', categorical_pipeline, categorical_cols)
])

preprocessor = Pipeline([
    ('brand_imputer', BrandImputer()),
    ('columns', column_preprocessor)
])

lr_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

ridge_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor', Ridge())
])

dt_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',DecisionTreeRegressor(random_state=42))
])

rf_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1))
])

### Price Band MAE Function

After observing fairly high MAE values across all models, I reviewed the largest errors and found that they were all on extremely expensive vehicles. Since the goal is to predict the values of all cars in the dataset, I decided not to drop the outliers. Instead, I am using the function below to evaluate how each model performs in each price band.

In [33]:
cv = KFold(n_splits = 5, shuffle=True, random_state=42)

def error_bands(model):
    cv_predictions = cross_val_predict(
    model, X_train, y_train, cv=cv
    )

    errors = pd.DataFrame({
        'actual': y_train.to_numpy(),
        'predicted': cv_predictions
    })

    errors['absolute_error'] = (
        errors['actual'] - errors['predicted']
    ).abs()

    errors['price_range'] = pd.cut(
        errors['actual'],
        bins = [-float("inf"), 20_000, 40_000, 60_000, 100_000, float("inf")],
        labels=['Under $20k','$20-40k','$40-60k','$60-100k','$100k+'],
        right=False
    )

    print(errors.groupby('price_range', observed=True).agg(
        cars=('actual','size'),
        mean_price=('actual','mean'),
        MAE=('absolute_error','mean'),
        median_error=('absolute_error','median')
    ).round(2))

## Linear Regression Model (Baseline)

In [34]:
lr_cv = cross_val_score(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

lr_pipeline.fit(X_train,y_train)

print(f"LR CV MAE: ${-lr_cv.mean():,.2f}")

LR CV MAE: $15,543.51


## Ridge Regression Model

### Untuned

In [35]:
ridge_cv = cross_val_score(
    ridge_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print(f"Ridge CV MAE: ${-ridge_cv.mean():,.2f}")

Ridge CV MAE: $13,694.62


### Tuned

In [36]:
ridge_param_grid = {
    'regressor__alpha':[0.01,.1,1,10,100,1000]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

print(f"Ridge Grid best alpha: {ridge_grid.best_params_}")
print(f"Ridge Grid best MAE: ${-ridge_grid.best_score_:,.2f}")

Ridge Grid best alpha: {'regressor__alpha': 0.1}
Ridge Grid best MAE: $13,036.06


In [37]:
error_bands(ridge_grid.best_estimator_)

             cars  mean_price       MAE  median_error
price_range                                          
Under $20k    967    12440.76   8736.83       6303.91
$20-40k      1046    29901.45   8546.52       5380.24
$40-60k       578    48438.49   8254.58       5390.57
$60-100k      389    75719.14  12617.92       8369.01
$100k+        190   206959.97  75034.87      30583.32


### Log Transformed Price Untuned

In [38]:
ridge_log = TransformedTargetRegressor(
    regressor=ridge_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

In [39]:
for name, model in [
    ('Original price', ridge_grid.best_estimator_),
    ('Log price', ridge_log)
]:
    scores=cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error'
    )

    print(f"{name} CV MAE: ${-scores.mean():,.2f}")

Original price CV MAE: $13,036.06
Log price CV MAE: $9,307.17


### Log Transformed Price Tuned

In [40]:
ridge_log_grid = GridSearchCV(
    ridge_log,
    param_grid={'regressor__regressor__alpha':[0.01,.1,1,10,100]},
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

ridge_log_grid.fit(X_train,y_train)

print(f"Ridge Grid best alpha: {ridge_log_grid.best_params_}")
print(f"Ridge Grid best MAE: ${-ridge_log_grid.best_score_:,.2f}")

Ridge Grid best alpha: {'regressor__regressor__alpha': 1}
Ridge Grid best MAE: $9,307.17


In [41]:
error_bands(ridge_log_grid.best_estimator_)

             cars  mean_price       MAE  median_error
price_range                                          
Under $20k    967    12440.76   2818.14       2210.23
$20-40k      1046    29901.45   4679.26       3548.28
$40-60k       578    48438.49   6491.63       4757.83
$60-100k      389    75719.14  11907.40       8736.81
$100k+        190   206959.97  71052.39      27129.00


## Decision Tree

### Untuned

In [42]:
dt_cv = cross_val_score(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print(f"Decision Tree CV MAE: ${-dt_cv.mean():,.2f}")

Decision Tree CV MAE: $13,391.13


### Tuned

In [43]:
dt_param_grid = {
    'regressor__max_depth':[10,20,None],
    'regressor__min_samples_split':[2,5,8],
    'regressor__min_samples_leaf':[1,2,4]
}

dt_grid = GridSearchCV(
    dt_pipeline,
    param_grid=dt_param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print(f"Decision Tree best params: {dt_grid.best_params_}")
print(f"Decision Tree best MAE: ${-dt_grid.best_score_:,.2f}")

Decision Tree best params: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 8}
Decision Tree best MAE: $12,871.81


In [44]:
error_bands(dt_grid.best_estimator_)

             cars  mean_price       MAE  median_error
price_range                                          
Under $20k    967    12440.76   4300.36       3182.00
$20-40k      1046    29901.45   7434.37       5547.75
$40-60k       578    48438.49  10855.61       6961.57
$60-100k      389    75719.14  19481.63      12401.67
$100k+        190   206959.97  79031.25      46016.50


### Log Transformed Price Untuned

In [45]:
dt_log = TransformedTargetRegressor(
    regressor=dt_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

for name, model in [
    ('Tuned Original price', dt_grid.best_estimator_),
    ('Log price', dt_log)
]:
    scores=cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error'
    )

    print(f"{name} CV MAE: ${-scores.mean():,.2f}")

Tuned Original price CV MAE: $12,871.81
Log price CV MAE: $13,989.29


### Log Transformed Price Tuned

In [46]:
dt_log_param_grid = {
    'regressor__regressor__max_depth':[10,20,None],
    'regressor__regressor__min_samples_split':[8,10,12],
    'regressor__regressor__min_samples_leaf':[1,2,4]
}

dt_log_grid = GridSearchCV(
    dt_log,
    param_grid=dt_log_param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

dt_log_grid.fit(X_train, y_train)

print(f"Decision Tree best params: {dt_log_grid.best_params_}")
print(f"Decision Tree best MAE: ${-dt_log_grid.best_score_:,.2f}")

Decision Tree best params: {'regressor__regressor__max_depth': 20, 'regressor__regressor__min_samples_leaf': 1, 'regressor__regressor__min_samples_split': 8}
Decision Tree best MAE: $12,809.20


In [47]:
error_bands(dt_log_grid.best_estimator_)

             cars  mean_price       MAE  median_error
price_range                                          
Under $20k    967    12440.76   4485.19       3223.12
$20-40k      1046    29901.45   7209.30       5286.63
$40-60k       578    48438.49  11807.75       7515.50
$60-100k      389    75719.14  19280.79      12856.04
$100k+        190   206959.97  75799.62      40407.94


## Random Forest

I opted not to test the Random Forest model on the log transformed price. 

### Untuned

In [48]:
rf_cv = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print(f"Random Forest CV MAE: ${-rf_cv.mean():,.2f}")

Random Forest CV MAE: $10,428.09


### Tuned

In [49]:
rf_param_grid = {
    'regressor__n_estimators':[400,550,700],
    'regressor__max_features':['sqrt',None],
    'regressor__min_samples_split':[2,4,8],
    'regressor__min_samples_leaf':[1,2],
    'regressor__max_depth':[10,20,None]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print(f"Random Forest best params: {rf_grid.best_params_}")
print(f"Random Forest best MAE: ${-rf_grid.best_score_:,.2f}")

Random Forest best params: {'regressor__max_depth': None, 'regressor__max_features': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 550}
Random Forest best MAE: $10,410.43


In [50]:
error_bands(rf_grid.best_estimator_)

             cars  mean_price       MAE  median_error
price_range                                          
Under $20k    967    12440.76   3428.16       2607.24
$20-40k      1046    29901.45   5466.13       4087.44
$40-60k       578    48438.49   7947.08       5681.06
$60-100k      389    75719.14  15531.37      12084.54
$100k+        190   206959.97  70175.48      32885.73


## Model Evaluation and Comparison

In [51]:
models = {
    "Linear Regression": lr_pipeline,
    "Ridge (log)": ridge_log_grid.best_estimator_,
    "Decision Tree": dt_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_,
}

model_order = ['Linear Regression','Ridge (log)','Decision Tree','Random Forest']

bins = [-float("inf"), 20_000, 40_000, 60_000, 100_000, float("inf")]
labels = ["Under $20k", "$20–40k", "$40–60k", "$60–100k", "$100k+"]

actual = pd.Series(y_train, index=X_train.index)
price_bands = pd.cut(actual, bins=bins, labels=labels, right=False)

results = []

for model_name, model in models.items():
    cv_predictions = cross_val_predict(
        model, X_train, y_train, cv=cv
    )

    band_data = pd.DataFrame({
        "Price band": price_bands.to_numpy(),
        "Actual price": y_train.to_numpy(),
        "Prediction": cv_predictions
    })
    band_data["Absolute error"] = (
    band_data["Actual price"] - band_data["Prediction"]
    ).abs()

    summary = (
        band_data.groupby("Price band", observed=False)
        .agg(Cars=("Absolute error", "size"),
    MAE=("Absolute error", "mean"))
    .reset_index()
    )
    summary["Model"] = model_name
    results.append(summary)

    mae_by_band = pd.concat(results, ignore_index=True)

    mae_table = mae_by_band.pivot(
    index="Price band", columns="Model", values="MAE"
    ).reindex(index=labels, columns=model_order).round(2)

mae_table


Model,Linear Regression,Ridge (log),Decision Tree,Random Forest
Price band,,,,
Under $20k,10672.62,2818.14,4300.36,3428.16
$20–40k,10160.78,4679.26,7434.37,5466.13
$40–60k,11045.22,6491.63,10855.61,7947.08
$60–100k,16820.62,11907.40,19481.63,15531.37
$100k+,81036.63,71052.39,79031.25,70175.48


I am selecting the Ridge model that had the price log transformed as my champion model. It outperformed all other models across all price bands, with the exception of the Random Forest in the $100k+ band.

In [52]:
cv_predictions = cross_val_predict(
    ridge_log_grid.best_estimator_,
    X_train,
    y_train,
    cv=cv
)

error_check = X_train.copy()
error_check['Actual Price'] = y_train.to_numpy()
error_check['Predicted Price'] = cv_predictions
error_check['Absolute Error'] = (
    error_check['Actual Price'] - error_check['Predicted Price']
).abs()

error_check.sort_values('Absolute Error', ascending=False).head(10)

,brand,model,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,vehicle_age,hp,liters,Actual Price,Predicted Price,Absolute Error
229,Bugatti,Veyron 16.4 Grand Sport,6330,Gasoline,8.0L W16 64V GDI DOHC Twin Turbo,7-Speed A/T with Auto-Shift,White,White,None reported,Yes,13,NaN,8.0,1950995,59082.988438,1.891912e+06
3046,Porsche,Carrera GT Base,4400,Gasoline,605.0HP 5.7L 10 Cylinder Engine Gasoline Fuel,6-Speed M/T,Gray,Black,None reported,Yes,19,605.0,5.7,1599000,98387.380780,1.500613e+06
1356,Lamborghini,Aventador SVJ Base,6987,Gasoline,6.5L V12 48V MPFI DOHC,7-Speed,Unknown,Nero Ade,None reported,Yes,3,NaN,6.5,749950,313185.251078,4.367647e+05
203,Ford,GT,1897,Gasoline,5.4 Liter SC DOHC,M/T,White,Unknown,None reported,Unknown,19,NaN,5.4,429998,28971.117582,4.010269e+05
2133,Cadillac,Escalade ESV Sport,4283,Gasoline,6.2L V8 16V GDI OHV,10-Speed A/T,Black,Brandy,None reported,Yes,3,NaN,6.2,479995,91887.411157,3.881076e+05
1148,Porsche,911 R,719,Gasoline,4.0L H6 24V GDI DOHC,6-Speed M/T,White,Black / Brown,None reported,Yes,8,NaN,4.0,489000,112764.737229,3.762353e+05
1061,Dodge,Viper GTC,1389,Gasoline,645.0HP 8.4L 10 Cylinder Engine Gasoline Fuel,M/T,Black,Black,None reported,Yes,7,645.0,8.4,489995,135526.877916,3.544681e+05
304,Rolls-Royce,Phantom Drophead Coupe Drophead,2958,Gasoline,6.8L V12 48V GDI DOHC,8-Speed A/T,Yellow,Unknown,None reported,Yes,11,NaN,6.8,488880,134925.842677,3.539542e+05
624,Rolls-Royce,Cullinan,398,Gasoline,6.7L V12 48V GDI DOHC Twin Turbo,8-Speed A/T,Silver,Charles Blue,None reported,Yes,2,NaN,6.7,695000,351832.528691,3.431675e+05
979,Lamborghini,Aventador SVJ Base,6929,Gasoline,759.0HP 6.5L 12 Cylinder Engine Gasoline Fuel,A/T,Blue,Black,None reported,Yes,5,759.0,6.5,649999,371003.418529,2.789956e+05


## Final Model Evaluation on Test Set

In [53]:
ridge_log_pred = ridge_log_grid.predict(X_test)
y_test_values = np.asarray(y_test)

ridge_log_test_results = pd.Series({
    'MAE': mean_absolute_error(y_test_values, ridge_log_pred),
    'RMSE': root_mean_squared_error(y_test_values, ridge_log_pred),
    'R^2': r2_score(y_test_values, ridge_log_pred)
}).round(2)

ridge_log_test_results

MAE      8466.27
RMSE    25752.15
R^2         0.74
dtype: float64

In [54]:
test_price_bands = pd.cut(y_test_values, bins=bins, labels=labels, right=False)

ridge_log_band_results = pd.DataFrame({
    'Price Band': test_price_bands,
    'Absolute error': np.abs(y_test_values - ridge_log_pred)
})

ridge_log_band_results.groupby(
    'Price Band', observed=False
).agg(
    Count=('Absolute error','size'),
    MAE=('Absolute error','mean')
).round(2)

,Count,MAE
Price Band,,
Under $20k,235,2721.67
$20–40k,263,4832.62
$40–60k,159,5326.50
$60–100k,94,11270.38
$100k+,42,68972.57


## Permutation Importance

In [55]:
ridge_permutation = permutation_importance(
    ridge_log_grid.best_estimator_,
    X_test,
    y_test,
    scoring='neg_mean_absolute_error',
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

ridge_importance = (
    pd.DataFrame({
        'Feature': X_test.columns,
        'Mean MAE increase': ridge_permutation.importances_mean,
        'Std': ridge_permutation.importances_std
    })
    .sort_values('Mean MAE increase', ascending=False)
    .reset_index(drop=True)
)

ridge_importance.round(2)

,Feature,Mean MAE increase,Std
0,brand,7699.60,421.72
1,vehicle_age,7557.46,251.74
2,hp,6446.63,337.26
3,milage,5833.25,279.26
4,liters,4015.78,266.42
5,model,2338.78,143.00
6,engine,2287.36,168.06
7,fuel_type,554.99,98.78
8,transmission,396.26,106.98
9,int_col,172.05,73.18


## Interpretation of Final Results

The final model performed similarly on the test set and cross validated training sets. The overall MAE is $8466.27. The model performs better for lower priced vehicles and has substantial errors for expensive vehicles. The R^2 score of 0.74 means that the predictor variables in the model accounted for 74% of the variation in the target variable (price). Permutation importances show that the model relied most heavily on vehicle age, horsepower, brand, and mileage to predict the price. Shuffling these features caused the largest increase in MAE.

### Limitations

Since the original listings for these vehicles are not available, I can't reliably rule out price values that I suspect are outliers, or obtain any of the missing information.